In [4]:
from datetime import datetime
from lightningrod.utils import config
from dotenv import load_dotenv
from lightningrod import (
    LightningRod,
    GdeltSeedGenerator,
    ForwardLookingQuestionGenerator,
    QuestionPipeline,
    QuestionRenderer,
    WebSearchLabeler,
    MultipleChoiceAnswerType,
    multiple_choice_example,
)

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

In [5]:
instructions = """
Generate multiple-choice forecasting questions about a specific future real-world event, based on recent news coverage.

Each question MUST resolve within the next three months.
"""

good_examples = [
    # 3 options
    multiple_choice_example(
        "What will the European Central Bank decide at its April 10, 2025 monetary policy meeting regarding its benchmark interest rate?",
        ["Rate increase", "No change", "Rate cut"],
        label=1,
    ),
    # 4 options
    multiple_choice_example(
        "By July 15, 2025, what will be the status of the proposed merger between Kroger and Albertsons?",
        ["Fully approved", "Regulator blocked", "Deal withdrawn", "Under review"],
        label=1,
    ),
    # 5 options
    multiple_choice_example(
        "By June 30, 2025, how many countries will have formally ratified the Global Plastics Treaty adopted in November 2024?",
        ["Fewer than 20", "20\u201339", "40\u201359", "60\u201379", "80 or more"],
        label=1,
    ),
    # 6 options
    multiple_choice_example(
        "By August 31, 2025, what stage will the United Kingdom's Sizewell C nuclear power project have reached?",
        ["Pre-construction", "Investment approved", "Site preparation", "Reactor construction", "Power generation", "Project canceled"],
        label=1,
    ),
]

bad_examples = [
    multiple_choice_example(
        "Which of the following will occur by December 31, 2025?",
        ["Japan raises interest rates", "Apple releases a foldable iPhone", "Brazil hosts a climate summit", "None of the above"],
        comment="Multiple unrelated events; violates single-event and IIA criteria.",
    ),
    multiple_choice_example(
        "What will Russia do by July 1, 2025 regarding Ukraine?",
        ["Launch a new offensive", "Launch a new offensive and mobilize additional troops", "Take no new military action"],
        comment="option_1 is a subset of option_0; logical nesting breaks IIA.",
    ),
    multiple_choice_example(
        "Will the merger between Company X and Company Y be approved by regulators by May 1, 2025?",
        ["Yes", "No", "Still under review", "Not reported"],
        comment="Binary yes/no disguised as multiple choice; mixes outcome with reporting status.",
    ),
    multiple_choice_example(
        "By October 2025, how will inflation in Argentina change?",
        ["Increase significantly", "Increase slightly", "Stay about the same", "Decrease"],
        comment="Vague, non-verifiable magnitude; multiple answers could be correct.",
    ),
    multiple_choice_example(
        "Which of the following will occur by January 31, 2025?",
        ["A major new sanctions package is announced", "A ceasefire agreement is signed", "Both option_0 and option_1", "Neither option_0 nor option_1"],
        comment="Combines independent events; explicit IIA violation.",
    ),
    multiple_choice_example(
        "Who will win the presidential election in the United States in 2024?",
        ["Donald Trump", "Kamala Harris"],
        comment="Only two options; use binary answer type instead.",
    ),
]

In [6]:
answer_type = MultipleChoiceAnswerType()

pipeline = QuestionPipeline(
    seed_generator=GdeltSeedGenerator(
        start_date=datetime(2024, 7, 1),
        end_date=datetime(2025, 11, 30),
        articles_per_interval=35,
        interval_duration_days=7,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        questions_per_seed=2,
        answer_type=answer_type,
    ),
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.9,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

dataset = lr.transforms.run(pipeline, max_questions=200)  # Increase to ~10000 for a real run

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $4.58                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ GdeltSeedGenerato… │ Complete             │   2 │  60 │        0 │      0 │ -                  │       3s │  │
│  │ ForwardLookingQue… │ Complete             │  60 │ 112 │        8 │      0 │ date_close not     │       9s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (8)                │          │  │
│  │ WebSearchLabelerT… │ Complete             │ 112 │ 110 │        2 │      0 │ Undetermined label │      53s │  │
│  │                    │                      │     │     │          │        │ (1), Low           │          │  │
│  │                    │                      │     │     │          │        │ confidence: 0.80 < │          │  │
│  │                    │                      │     │     │          │        │ 0.9 (1)            │          │  │
│  │ QuestionRendererT… │ Complete             │ 110 │ 110 │        0 │      0 │ -                  │       0s │  │
│  └────────────────────┴──────────────────────┴─────┴─────┴──────────┴────────┴────────────────────┴──────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
import pandas as pd

pd.DataFrame(dataset.flattened())

,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,prompt,seed_text,seed_url,seed_creation_date,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms,invalid_reason
0,02174d89-9caa-46e6-b71a-d3eedbb3d4c2,True,What will be the operational status of the Isr...,2025-10-31T00:00:00,2025-10-04T00:00:00,This question resolves based on the official m...,2025-10-04T00:00:00,Temporary ceasefire but maintaining active com...,multiple_choice,0.90,...,"By October 31, 2025, the IDF maintained a stat...",https://vertexaisearch.cloud.google.com/ground...,QUESTION:\nWhat will be the operational status...,Israel's army said Saturday that it would adva...,https://economictimes.indiatimes.com/news/inte...,2025-10-04T00:00:00,25dae081-96a7-4be4-a120-db9fc2efe7cc,55dbf5b6-1c8a-45a6-9cc0-4e3cc9f15cb5,90021.965,NaN
1,05321453-f280-46fb-b47b-c0b83cd6d10c,True,"On January 20, 2025, will Melania Trump be pre...",2025-01-21T00:00:00,2024-11-06T00:00:00,This question will resolve based on visual evi...,2024-11-06T00:00:00,"Yes, she is present",multiple_choice,1.00,...,Donald Trump was inaugurated as the 47th Presi...,https://vertexaisearch.cloud.google.com/ground...,"QUESTION:\nOn January 20, 2025, will Melania T...",The First Meeting and First Date\nDonald Trump...,https://economictimes.indiatimes.com/magazines...,2024-11-06T00:00:00,e75b08c5-bc18-44f7-ac34-d20df414eec3,44ddcf55-7ae1-411b-8fdd-6c498dcaaea7,20007.691,NaN
2,07566a2d-6fa6-4f25-828b-90365e5c3f13,True,Whom will President-elect Donald Trump formall...,2024-12-31T00:00:00,2024-11-06T00:00:00,The question resolves based on the first indiv...,2024-11-06T00:00:00,Scott Bessent,multiple_choice,1.00,...,President-elect Donald Trump officially announ...,https://vertexaisearch.cloud.google.com/ground...,QUESTION:\nWhom will President-elect Donald Tr...,"In the coming weeks, President-elect Trump cou...",https://economictimes.indiatimes.com/news/inte...,2024-11-06T00:00:00,0b99545c-d6b2-4afb-b440-644583f5bc38,a23a9037-41c8-4de8-9d60-c81e725eae76,26053.247,NaN
3,088cd9ea-0d17-484d-81e9-87afb2f4ec44,False,"By July 1, 2025, will the government of Taiwan...",2025-07-01T00:00:00,2025-10-01T00:00:00,Resolution is based on the signing of a bilate...,2025-10-01T00:00:00,NaN,NaN,NaN,...,NaN,None,NaN,"Taiwan ""will not agree"" to making 50 percent o...",https://economictimes.indiatimes.com/news/inte...,2025-10-01T00:00:00,733ad05b-2f01-4cf0-bb03-f5287348e040,c6972913-47c6-451f-b936-34b5585ad06d,1131.197,date_close not after event_date
4,0ee7dc1f-c66c-4300-8805-e799a2e98610,False,"By April 15, 2025, will the Indian Ministry of...",2025-04-15T00:00:00,2025-10-01T00:00:00,This question will be resolved by checking the...,2025-10-01T00:00:00,NaN,NaN,NaN,...,NaN,None,NaN,सरकार ने इलेक्ट्रिक गाड़ियों में सुरक्षा के लि...,https://navbharattimes.indiatimes.com/auto/car...,2025-10-01T00:00:00,2f5c0e21-0f8e-4174-95d2-762949882fb4,23907061-2917-480f-89c7-1dc3dc057e23,192.318,date_close not after event_date
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,f81e3abc-6f2a-4621-a8f8-7688fdb5e696,True,"By February 15, 2025, what will be the officia...",2025-02-15T00:00:00,2024-11-06T00:00:00,The question will be resolved by official publ...,2024-11-06T00:00:00,Nomination confirmed,multiple_choice,1.00,...,"The close date: 2025-02-15, the question date:...",https://vertexaisearch.cloud.google.com/ground...,"QUESTION:\nBy February 15, 2025, what will be ...","Internally, Trump has already indicated his ad...",https://economictimes.indiatimes.com/news/inte...,2024-11-06T00:00:00,75bbee22-48cf-484f-8a20-1c98965b9128,0b5215f4-ba42-47e2-b0eb-e480791aa4ef,89138.057,NaN
116,f85959b3-b20c-411c-a362-a61e9d3decf8,True,"By December 1, 2025, what will be the official...",2025-12-01T00:00:00,2025-10-04T00:00:00,This question will be resolved based on offici...,2025-10-04T00:00:00,Complete 